# Prompt Chaining and Task Decomposition

A single prompt asking an LLM to "read this document, extract entities, classify sentiment, draft a reply, and format JSON" often fails — context gets crowded, steps get skipped, and errors are hard to debug. **Task decomposition** splits work into smaller subtasks. **Prompt chaining** runs those subtasks **sequentially**, passing each step's output as input to the next.

Chaining is the simplest multi-step LLM pattern: no special framework required — just multiple API calls and glue code. It pairs naturally with **structured output** (notebook 06) so each step returns parseable data for the next.

This notebook covers when to decompose tasks, sequential chain design, common patterns (summarize→extract, plan→execute, draft→critique), trade-offs in latency and error propagation, and live demonstrations building a support-ticket pipeline in Python.

Prerequisites: **01–06** in this section and basic API usage from **03. LLM API Integration**.

---

## 1. Breaking Down Complex Tasks

### 1.1 When One Prompt Is Not Enough

| Signal | Example |
|--------|---------|
| **Multiple distinct goals** | Summarize + extract dates + classify urgency |
| **Long input** | 20-page contract; model loses focus |
| **Mixed reasoning types** | Math verification then creative rewrite |
| **Quality gate** | Draft must pass a checklist before sending |
| **Tool boundaries** | Search docs, then answer from results only |
| **Unreliable monolith** | One-shot JSON often missing fields |

Decomposition trades **more API calls** for **higher per-step reliability** and **easier debugging** (you know which step failed).

### 1.2 Task Decomposition vs Prompt Chaining

| Concept | Meaning |
|---------|---------|
| **Task decomposition** | Design: breaking a problem into subtasks |
| **Prompt chaining** | Implementation: run subtasks in order, wire outputs |

Decomposition is the plan; chaining is one execution strategy. Alternatives include **parallel branches** (run independent subtasks, merge) and **routing** (classifier picks which chain to run) — covered in advanced architecture topics.

### 1.3 The Chain Mental Model

```
Input ──► Step A (prompt) ──► output_A ──► Step B (prompt + output_A) ──► output_B ──► ...
```

Each step should have:

- **One clear objective** ("extract order ID" not "do everything")
- **Defined input** (original text, prior step output, or both)
- **Defined output shape** (prose, JSON, bullet list)
- **Validation** before passing to the next step

---

## 2. Sequential Prompt Chains

### 2.1 Passing Output Between Steps

Three common wiring patterns:

| Pattern | What step B receives |
|---------|----------------------|
| **Replace** | Only output from step A (A compressed the input) |
| **Append** | Original input + output from step A |
| **Structured handoff** | Parsed JSON from step A injected into B's template |

Example append pattern:

```
Step B user message:
Facts extracted:
{output_a}

Original email:
{original}

Draft a reply addressing only the facts above.
```

### 2.2 Stateless Steps

Each chain step is typically a **fresh** API call with a minimal message list — you do not need full chat history unless the step benefits from it. Pass only what the next prompt requires to save tokens.

### 2.3 Human-in-the-Loop Checkpoints

Insert approval between steps in production:

1. Extract structured ticket fields → show UI for edit
2. User confirms → generate customer reply

Chains are not always fully automatic.

---

## 3. Chain Design Patterns

### 3.1 Summarize → Extract

Long document → short summary → structured fields from summary.

- **Why:** Summary reduces noise; extraction prompt stays small
- **Risk:** Summary may drop rare but critical details (dates, IDs)
- **Mitigation:** Extract from summary **and** quote relevant spans from original

### 3.2 Plan → Execute

Step 1: outline steps or sub-questions. Step 2: answer each part. Step 3 (optional): merge.

- **Why:** Separates strategy from execution (related to CoT, notebook 04)
- **Use:** Research briefs, multi-section reports, debugging workflows

### 3.3 Classify → Route → Specialized Prompt

Step 1: category label. Step 2: run category-specific prompt (billing vs technical).

- **Why:** Each branch prompt is simpler and more accurate
- **Use:** Support bots, document routers

### 3.4 Draft → Critique → Revise

Step 1: first draft. Step 2: critic lists issues against rubric. Step 3: revision.

- **Why:** Improves quality without one overloaded instruction
- **Cost:** 3× calls; use when quality matters (legal summaries, public comms)

### 3.5 Retrieve → Answer (RAG chain)

Step 1: embed query, fetch chunks. Step 2: answer using chunks only.

- **Why:** Grounds answers in sources
- **Note:** Retrieval is often non-LLM code; generation is the second link

---

## 4. Trade-offs

### 4.1 Latency and Cost

A chain with $N$ steps roughly multiplies:

- **Wall-clock time** — steps run in series (unless parallelized)
- **Token spend** — each step has its own prompt + completion

| Chain length | Typical use |
|--------------|-------------|
| 2 steps | Extract → respond |
| 3 steps | Draft → critique → revise |
| 4+ steps | Complex pipelines; consider caching intermediate results |

### 4.2 Error Propagation

If step 1 extracts the wrong order ID, step 2 drafts a reply about the wrong order. Errors **compound** downstream.

**Mitigations:**

- Validate step outputs (regex, JSON schema, Pydantic)
- Use **low temperature** on extraction/classification steps
- Pass **original source** to later steps, not only prior output
- Add a **verification** step ("does the reply cite facts present in the email?")
- Log each step's input/output for debugging

### 4.3 When to Stay Monolithic

One prompt is fine when:

- Task is short and single-purpose
- Latency budget is tight
- Model reliably completes the combined task in testing

Start monolithic; decompose when quality or maintainability suffers.

---

## 5. Demonstration Setup

Sample customer email for all chain demos. Replace the placeholder API key before running.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"

CUSTOMER_EMAIL = """Subject: Wrong item + late delivery

Hi,
I ordered blue running shoes (order #ORD-88291) on March 3. They arrived today
but I received red sneakers instead. I need the correct item before my marathon
on March 20. Please advise on exchange and whether you can expedite shipping.

Thanks,
Priya"""


def llm(system: str, user: str, max_tokens: int = 300, json_mode: bool = False) -> str:
  kwargs = {
      "model": MODEL,
      "messages": [
          {"role": "system", "content": system},
          {"role": "user", "content": user},
      ],
      "temperature": 0.0,
      "max_tokens": max_tokens,
  }
  if json_mode:
      kwargs["response_format"] = {"type": "json_object"}
  response = client.chat.completions.create(**kwargs)
  return response.choices[0].message.content or ""


print("Setup complete.")

---

## 6. Demonstration: Monolith vs Chain

**Monolith:** one prompt does extraction + classification + reply. **Chain:** three focused steps.

In [ ]:
monolith_system = "You are support staff. Be concise and professional."
monolith_user = f"""Read this email. In one response:
1) List key facts (order id, issue, deadline)
2) Classify urgency: LOW, MEDIUM, or HIGH
3) Draft a 3-sentence reply

Email:
{CUSTOMER_EMAIL}"""

print("=== MONOLITH (1 call) ===")
print(llm(monolith_system, monolith_user, max_tokens=400))

In [ ]:
# Step 1: Extract facts as JSON
extract_system = "Extract support ticket facts. Return JSON only."
extract_user = f"""Return JSON with keys:
- order_id (string or null)
- issue_summary (string)
- customer_deadline (string or null)
- items_mentioned (array of strings)

Email:
{CUSTOMER_EMAIL}"""

facts_json = llm(extract_system, extract_user, max_tokens=200, json_mode=True)
facts = json.loads(facts_json)
print("=== STEP 1: EXTRACT ===")
print(json.dumps(facts, indent=2))

# Step 2: Classify urgency from facts only
classify_system = "Classify support urgency. Reply with one word: LOW, MEDIUM, or HIGH."
classify_user = f"Facts:\n{json.dumps(facts, indent=2)}"
urgency = llm(classify_system, classify_user, max_tokens=10).strip()
print("\n=== STEP 2: CLASSIFY ===")
print(urgency)

# Step 3: Draft reply using facts + original email
reply_system = "You are a support agent. Write a professional 3-sentence reply."
reply_user = f"""Urgency: {urgency}
Facts: {json.dumps(facts)}

Original email:
{CUSTOMER_EMAIL}

Draft the reply."""
reply = llm(reply_system, reply_user, max_tokens=250)
print("\n=== STEP 3: REPLY ===")
print(reply)

The chain version logs intermediate artifacts (`facts`, `urgency`) you can store in a ticket system. The monolith is faster but harder to validate programmatically.

---

## 7. Demonstration: Summarize → Extract

For long inputs, summarize first — then extract structured fields from the summary **plus** a validation pass on the original.

In [ ]:
LONG_EMAIL = CUSTOMER_EMAIL + "\n\n" + (
    "P.S. I also want to mention that your website checkout was slow and I had "
    "to refresh twice. Not the main issue but worth fixing."
)

summary = llm(
    "Summarize customer emails in 3 bullet points. Focus on the main problem.",
    LONG_EMAIL,
    max_tokens=150,
)
print("=== SUMMARY ===")
print(summary)

extract_from_summary = llm(
    "From the summary, return JSON with keys: primary_issue, order_id, deadline.",
    f"Summary:\n{summary}",
    max_tokens=120,
    json_mode=True,
)
print("\n=== EXTRACT FROM SUMMARY ===")
print(json.dumps(json.loads(extract_from_summary), indent=2))

The P.S. about checkout may disappear from the summary — acceptable if your pipeline only handles product issues; otherwise include full text in a later step.

---

## 8. Demonstration: Plan → Execute

Step 1 produces an outline; step 2 writes one section at a time (here simplified to plan + single execution).

In [ ]:
plan = llm(
    "Create a numbered action plan for support to resolve the ticket. Max 4 steps.",
    f"Customer email:\n{CUSTOMER_EMAIL}",
    max_tokens=200,
)
print("=== PLAN ===")
print(plan)

execution = llm(
    "Write internal support notes following the plan exactly. Use bullet points.",
    f"Plan:\n{plan}\n\nCustomer email:\n{CUSTOMER_EMAIL}",
    max_tokens=300,
)
print("\n=== INTERNAL NOTES (EXECUTE) ===")
print(execution)

---

## 9. Demonstration: Draft → Critique → Revise

Three-step quality chain. The critic uses a explicit rubric.

In [ ]:
draft = llm(
    "Draft a brief customer reply.",
    CUSTOMER_EMAIL,
    max_tokens=200,
)
print("=== DRAFT ===")
print(draft)

critique = llm(
    "You are a QA reviewer. Score the draft 1-5 on: acknowledges wrong item, "
    "mentions exchange, addresses deadline. List specific fixes needed.",
    f"Email:\n{CUSTOMER_EMAIL}\n\nDraft:\n{draft}",
    max_tokens=200,
)
print("\n=== CRITIQUE ===")
print(critique)

revised = llm(
    "Revise the draft using the critique. Output only the final reply.",
    f"Original draft:\n{draft}\n\nCritique:\n{critique}\n\nEmail:\n{CUSTOMER_EMAIL}",
    max_tokens=250,
)
print("\n=== REVISED ===")
print(revised)

---

## 10. Packaging a Chain in Python

Production code wraps steps in a function with logging and validation hooks.

In [ ]:
def support_chain(email: str) -> dict:
    """Extract → classify → reply. Returns all step outputs for auditing."""
    facts_raw = llm(
        "Extract ticket facts as JSON. Keys: order_id, issue_summary, customer_deadline.",
        email,
        max_tokens=200,
        json_mode=True,
    )
    facts = json.loads(facts_raw)

    urgency = llm(
        "Reply with one word: LOW, MEDIUM, or HIGH.",
        json.dumps(facts),
        max_tokens=10,
    ).strip()

    reply = llm(
        "Professional 3-sentence support reply.",
        f"Urgency: {urgency}\nFacts: {json.dumps(facts)}\n\nEmail:\n{email}",
        max_tokens=250,
    )

    return {"facts": facts, "urgency": urgency, "reply": reply}


result = support_chain(CUSTOMER_EMAIL)
print(json.dumps({"facts": result["facts"], "urgency": result["urgency"]}, indent=2))
print("\nReply:", result["reply"][:200], "...")

---

## 11. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Too many steps | Slow, expensive | Merge steps that always succeed together |
| Only passing prior output | Lost details from source | Include original where needed |
| No validation between steps | Silent error cascade | JSON schema / rules per step |
| Same temperature everywhere | Creative extraction errors | Low temp for extract/classify |
| Giant prompts per step | Token waste | Compress earlier outputs |
| No logging | Hard to debug | Store step I/O with request IDs |

---

## 12. Summary

**Task decomposition** splits complex work into focused subtasks. **Prompt chaining** runs them sequentially, passing outputs forward — often with structured JSON handoffs between steps.

Common patterns: **summarize→extract**, **plan→execute**, **classify→route**, **draft→critique→revise**, and **retrieve→answer**. Chains improve debuggability and per-step accuracy at the cost of **latency**, **tokens**, and **error propagation**.

Demonstrations compared monolithic vs three-step support pipelines, summarize-then-extract, plan-execute, draft-critique-revise, and a reusable `support_chain()` function.

Next: **08. Prompt Patterns and Advanced Techniques** — ReAct, tree-of-thought, meta-prompting, and other reusable patterns.